In [ ]:
from openai import OpenAI
import os
import json
import datetime

client = OpenAI(api_key = "<api_key>")

In [ ]:
batch_ids = {}

current_time = datetime.datetime.strftime(datetime.datetime.now(), '%Y-%m-%d_%H:%M:%S')

for dataset in ['type1', 'type2', 'type3']:
    batches = os.listdir(f'GPT_batches/{dataset}')
    for i in range(len(batches)):
        batch_input_file = client.files.create(
            file=open(f"GPT_batches/{dataset}/{i}.jsonl", "rb"),
            purpose="batch"
        )
        batch = client.batches.create(
            input_file_id = batch_input_file.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={
                "description": f"{dataset}_batch_c2t_{i}",
            }
        )
        if dataset not in batch_ids:
            batch_ids[dataset] = []
        batch_ids[dataset].append(batch.id)

        print(f"Created batch {batch.id} for {dataset}_batch_{i} with file {[i]}.jsonl")
        
with open(f'batch_ids_{current_time}.json', 'w') as f:
    json.dump(batch_ids, f)